<div style="background:linear-gradient(135deg,#1b1448 0%,#3b1f5e 55%,#7c2d12 100%);border-radius:18px;padding:34px 30px;color:#fff;font-family:Inter,Segoe UI,sans-serif">
  <div style="font-size:12px;letter-spacing:3px;color:#fed7aa;font-weight:700;text-transform:uppercase">Chapter 3 · Solutions</div>
  <div style="font-size:34px;font-weight:900;line-height:1.1;margin:10px 0 6px">Tokens, context and cost · Solutions</div>
  <div style="font-size:15px;color:#ffe4d6;max-width:740px;line-height:1.6">Worked answers to the chapter's three practice challenges.</div>
  <div style="font-size:13px;color:#fed7aa;margin-top:14px">Building AI with AI: A Visual Handbook · John Fisher</div>
</div>

Worked answers to Chapter 3's practice challenges. The first counts tokens with `count_tokens`, and the other two use only the module from the chapter's Claude Code build.

## Setup

The same Setup as the chapter's notebook, with this notebook's own recording, and the module from the chapter's Claude Code build.

In [1]:
# Set LIVE = True to send these requests to the Anthropic API with your own key, saved in
# Colab's secrets panel as ANTHROPIC_API_KEY. Left False, the notebook replays the answers
# recorded for the book, with no key and no cost.
LIVE = False
%pip install -q --disable-pip-version-check anthropic==2.7.0
import importlib, pathlib, sys, urllib.request
if "google.colab" in sys.modules:
    BASE = "https://raw.githubusercontent.com/johnfisher-ai/ai-with-ai/main/chapters/notebooks/"
    pathlib.Path("recordings").mkdir(exist_ok=True)
    for name in ["book_client.py", "recordings/tokens-context-and-cost--solutions.json"]:
        urllib.request.urlretrieve(BASE + name, name)
import book_client
importlib.reload(book_client)
client = book_client.make_client("tokens-context-and-cost--solutions", mode="live" if LIVE else None)

Note: you may need to restart the kernel to use updated packages.


Replaying 3 recorded responses from claude-sonnet-5-5, recorded 2026-11-12. Set LIVE = True in Setup to call the API yourself.


In [2]:
# tokencost.py is the module Claude Code generated in this chapter's build, from the spec and tests in
# builds/tokens-context-and-cost/. Colab fetches it from the book's repository.
BUILD = "tokens-context-and-cost"
if "google.colab" in sys.modules:
    urllib.request.urlretrieve("https://raw.githubusercontent.com/johnfisher-ai/ai-with-ai/main/builds/"
                               f"{BUILD}/generated/tokencost.py", "tokencost.py")
else:
    sys.path.insert(0, str(pathlib.Path("../../builds", BUILD, "generated").resolve()))
import tokencost
tokencost = importlib.reload(tokencost)
from tokencost import PRICES, PRICES_AS_OF, call_cost, context_check, project_conversation, prompt_tokens

# The whole module, so you can read every function before the notebook uses it
import inspect
print(inspect.getsource(tokencost))

PRICES_AS_OF = "2026-11"

PRICES = {
    "claude-haiku-5-5": {"input": 0.10, "cache_write": 0.125, "cache_read": 0.01, "output": 0.50},
    "claude-sonnet-5-5": {"input": 2.00, "cache_write": 2.50, "cache_read": 0.10, "output": 10.00},
    "claude-opus-5-5": {"input": 4.00, "cache_write": 5.00, "cache_read": 0.20, "output": 20.00},
}

LONG_PROMPT_PRICES = {
    "claude-haiku-5-5": (
        100_000,
        {"input": 0.50, "cache_write": 0.625, "cache_read": 0.05, "output": 2.50},
    ),
}

_FIELDS = ("input_tokens", "output_tokens", "cache_read_input_tokens", "cache_creation_input_tokens")


def _counts(usage):
    out = {}
    for f in _FIELDS:
        if isinstance(usage, dict) or hasattr(usage, "get") and not hasattr(usage, f):
            v = usage.get(f)
        else:
            v = getattr(usage, f, None)
        v = 0 if v is None else v
        if v < 0:
            raise ValueError(f"{f} must not be negative")
        out[f] = v
    return out


def prompt_tokens(usage):
   

In [3]:
MODEL = "claude-sonnet-5-5"
TICKET = "My order arrived with a cracked screen. I'd like a replacement sent to the same address, please."

## Challenge 1. Separate the overhead from the text

Count the chapter's English ticket once, ten times and a hundred times, one copy per line, and work out what each copy adds and what the request costs before any text.

In [4]:
# Challenge 1. The ticket once, ten times and a hundred times, one copy per line.
sizes = {}
for copies in [1, 10, 100]:
    text = "\n".join([TICKET] * copies)
    sizes[copies] = client.messages.count_tokens(model=MODEL,
                                                 messages=[{"role": "user", "content": text}]).input_tokens
    print(f"{copies:>3} copies: {sizes[copies]:>5} input tokens")
per_copy = (sizes[100] - sizes[10]) / 90
print(f"each extra copy adds {per_copy:.1f} tokens, and the request's fixed overhead is about "
      f"{sizes[1] - per_copy:.0f}")

  1 copies:    38 input tokens
 10 copies:   317 input tokens
100 copies:  3107 input tokens
each extra copy adds 31.0 tokens, and the request's fixed overhead is about 7


Each copy after the first adds 31.0 tokens, two more than the 29 the chapter measured for the ticket alone, which the line break between copies accounts for. The single ticket counts 38, so the request's own overhead is 7 tokens if the line break is counted with the text and 9 if it isn't, the 9 the chapter measured with a one-character request. Counts grow linearly with the text, so a rate measured per copy prices a larger batch of the same kind of text.

## Challenge 2. When the history costs more than the reply

Output tokens cost five times as much as input on Claude Sonnet 5.5, so a short conversation spends most of its money on replies. Use `project_conversation` with the chapter's measured pattern to find the first turn whose input costs more than its output.

In [5]:
# Challenge 2. The chapter's conversation: a first prompt of 53 tokens, about 173 tokens added a turn and
# about 159 output tokens a reply. Find the first turn whose input costs more than its output.
rates = PRICES[MODEL]
for k, turn in enumerate(project_conversation(53, 173, 159, 30), 1):
    cost_in = turn["input_tokens"] * rates["input"]
    cost_out = turn["output_tokens"] * rates["output"]
    if cost_in > cost_out:
        print(f"turn {k}: {turn['input_tokens']} input tokens cost more than {turn['output_tokens']} output tokens")
        break

turn 6: 918 input tokens cost more than 159 output tokens


From turn 6, with a 918-token prompt, each turn's input costs more than its 159-token reply. The crossover comes where the prompt passes five times the reply, 795 tokens, and every later turn tilts further toward input, the cost that Chapter 118, Caching and batching, reduces.

## Challenge 3. Price a batch job

The Batch API answers requests asynchronously at half the standard price on input and output, as of November 2026. Write `batch_cost` on top of `call_cost`, and a test that checks it against Claude Sonnet 5.5's halved rates.

In [6]:
# Challenge 3. The Batch API charges half price on input and output, as of November 2026.
BATCH_DISCOUNT = 0.5

def batch_cost(usage, model):
    """The cost of one call sent through the Batch API."""
    return call_cost(usage, model) * BATCH_DISCOUNT

def test_batch_cost_is_half_the_standard_cost():
    usage = {"input_tokens": 2_000, "output_tokens": 500}
    assert abs(batch_cost(usage, "claude-sonnet-5-5") - (2_000 * 1.00 + 500 * 5.00) / 1_000_000) < 1e-12

test_batch_cost_is_half_the_standard_cost()
print("pass:", batch_cost({"input_tokens": 2_000, "output_tokens": 500}, "claude-sonnet-5-5"), "dollars")

pass: 0.0045 dollars


The test passes, and 2,000 input tokens with 500 output tokens cost 0.0045 dollars through the Batch API, against 0.009 at the standard rates. The discount scales the whole cost, so `batch_cost` needs no rates of its own. Add the test to the build's `given/tests/` and the next build has to keep it passing.

---
<div style="text-align:center;color:#666f8e;font-size:12px;margin-top:10px">Building AI with AI: A Visual Handbook · © 2026 John Fisher · text CC BY 4.0, code MIT</div>